# الفصل 40: النماذج الموحّدة: Any-to-Any

> كل ما سبق (VLM، Diffusion، Speech) كانت لبنات بناء منفصلة.

- **الجزء:** 8 — النماذج متعددة الوسائط
- **الفصل:** 40
- **المستوى:** خبير
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 39: الصوت والكلام متعدد الوسائط
- **الوسوم:** Unified Multimodal، Chameleon، GPT-4o، Gemini

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch40-unified-multimodal.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch40-unified-multimodal.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch40-unified-multimodal.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch40-unified-multimodal.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch40-unified-multimodal.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch40-unified-multimodal.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** لا توجد مكتبات خارجية مطلوبة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [

]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch40-unified-multimodal.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch40-unified-multimodal.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch40-unified-multimodal.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch40-unified-multimodal.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch40-unified-multimodal.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch40-unified-multimodal.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> كل ما سبق (VLM، Diffusion، Speech) كانت لبنات بناء منفصلة.

> **بيانات الفصل**
>
> - **الجزء:** 8 — النماذج متعددة الوسائط
> - **المستوى:** خبير
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 39: الصوت والكلام متعدد الوسائط


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم الفكرة المحورية: "التوكنة هي كل ما تحتاجه"
- فهم تدريب نموذج موحّد — نفس هدف GPT تمامًا
- فهم Unified Multimodal


## 1. لماذا يهمّك هذا الموضوع؟

كل ما سبق (VLM، Diffusion، Speech) كانت لبنات بناء منفصلة. الاتجاه الحالي الأكثر تقدّمًا (Gemini، GPT-4o، Chameleon من Meta) يسعى لبناء **نموذج واحد موحّد** يتعامل مع كل الوسائط (نص، صورة، صوت، فيديو) بنفس آلية Transformer، بدخل وخرج من **أي** وسيط إلى **أي** وسيط.


## 2. الخريطة الذهنية


```text
النهج التقليدي (Modular):          النهج الموحَّد (Unified Any-to-Any):
Vision Encoder ─┐                    [محوّل ترميز موحّد]
Audio Encoder ──┼─> LLM منفصل         كل الوسائط → رموز منفصلة (tokens)
Text ───────────┘                    في vocabulary واحد مشترك
                                              │
                                      [Transformer واحد يتعامل
                                       مع كل الرموز بنفس الطريقة]
                                              │
                                      يُنتج: نص أو صورة أو صوت
                                      (فك ترميز حسب نوع الرمز)
```


## 3. الشرح المتدرج

### 🟢 أ) الفكرة المحورية: "التوكنة هي كل ما تحتاجه"

إذا استطعت تحويل **أي** وسيط إلى تسلسل من الرموز المنفصلة (tokens) من مفردات (vocabulary) مشتركة، فإن Transformer القياسي (نفس ما بنيناه في الفصل 11!) يستطيع التعامل مع المدخلات والمخرجات متعددة الوسائط دون أي تعديل معماري جوهري — فقط embedding table أكبر، وطبقة إخراج أكبر.


In [ ]:
class UnifiedMultimodalVocabulary:
    """
    توضيح مفاهيمي: كيف نُركّب vocabulary موحّد من عدة وسائط
    (نهج مشابه لما استخدمته Chameleon من Meta)
    """
    def __init__(self, text_vocab_size=32000,
                 image_codebook_size=8192,    # من VQ-GAN
                 audio_codebook_size=1024):   # من RVQ (الفصل 39)
        self.text_range = (0, text_vocab_size)
        self.image_range = (text_vocab_size, text_vocab_size + image_codebook_size)
        self.audio_range = (
            text_vocab_size + image_codebook_size,
            text_vocab_size + image_codebook_size + audio_codebook_size
        )
        self.total_vocab_size = self.audio_range[1]

        # رموز خاصة لتحديد بداية/نهاية كل وسيط ضمن التسلسل
        self.special_tokens = {
            "<img_start>": self.total_vocab_size,
            "<img_end>": self.total_vocab_size + 1,
            "<audio_start>": self.total_vocab_size + 2,
            "<audio_end>": self.total_vocab_size + 3,
        }
        self.total_vocab_size += len(self.special_tokens)

    def encode_interleaved_sequence(self, text_tokens, image_tokens=None, audio_tokens=None):
        """
        بناء تسلسل متداخل (interleaved) من عدة وسائط
        مثال: "صف هذه الصورة: <img_start>...<img_end> ثم استمع: <audio_start>...<audio_end>"
        """
        sequence = text_tokens[:]
        if image_tokens is not None:
            # إزاحة رموز الصورة لتقع ضمن نطاقها المخصص في الـ vocabulary الموحّد
            shifted_image_tokens = [t + self.image_range[0] for t in image_tokens]
            sequence += [self.special_tokens["<img_start>"]] + shifted_image_tokens + \
                        [self.special_tokens["<img_end>"]]
        if audio_tokens is not None:
            shifted_audio_tokens = [t + self.audio_range[0] for t in audio_tokens]
            sequence += [self.special_tokens["<audio_start>"]] + shifted_audio_tokens + \
                        [self.special_tokens["<audio_end>"]]
        return sequence


> 💡 **كيف نحصل على `image_tokens` منفصلة؟** عبر **VQ-VAE/VQ-GAN** — نفس فكرة RVQ من الفصل 39 لكن مطبّقة على patches الصورة بدلاً من الإطارات الصوتية: كل patch من الصورة يُقرَّب لأقرب متجه في codebook متعلَّم، فنحصل على فهرس (index) صحيح لكل patch، أي "رمز صورة" تمامًا كرمز BPE النصي.

### 🟡 ب) تدريب نموذج موحّد — نفس هدف GPT تمامًا

بمجرد توحيد التمثيل، التدريب يصبح **بالضبط** نفس هدف Next-Token Prediction الذي رأيناه في الفصل 12:


In [ ]:
class UnifiedAnyToAnyTransformer(nn.Module):
    """
    نموذج موحّد مفاهيميًا — نفس MiniGPT من الفصل 12
    لكن بمفردات (vocabulary) موسَّعة لتشمل رموز الصورة والصوت
    """
    def __init__(self, unified_vocab_size, d_model=4096, n_layers=32, n_heads=32, max_seq_len=8192):
        super().__init__()
        self.token_embedding = nn.Embedding(unified_vocab_size, d_model)
        # استخدام ModernTransformerBlock من الفصل 17 (RoPE + GQA + RMSNorm + SwiGLU)
        self.blocks = nn.ModuleList([
            ModernTransformerBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.output_head = nn.Linear(d_model, unified_vocab_size, bias=False)
        self.output_head.weight = self.token_embedding.weight  # Weight Tying (الفصل 12)

    def forward(self, unified_token_ids):
        x = self.token_embedding(unified_token_ids)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        return self.output_head(x)

    def generate_any_modality(self, prompt_ids, vocab_ranges, max_new_tokens=512):
        """
        التوليد يحدث تلقائيًا بأي وسيط حسب أي نطاق رموز يختاره النموذج!
        إذا بدأ بتوليد رمز ضمن image_range، فك ترميزه عبر VQ-GAN decoder
        إذا بدأ بتوليد رمز ضمن text_range، فك ترميزه عبر tokenizer النصي
        """
        generated = prompt_ids.clone()
        for _ in range(max_new_tokens):
            logits = self.forward(generated)[:, -1, :]
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if next_token.item() in [vocab_ranges["eos_token_id"]]:
                break
        return generated


> ⚠️ **خطأ شائع**: الاعتقاد بأن هذا النهج "سهل" لمجرد بساطته المعمارية. التحدي الحقيقي هو **موازنة البيانات** بين الوسائط أثناء التدريب (Data Mixing)، لأن نسب بيانات الصورة/الصوت/النص المتفاوتة بشدة تؤدي لهيمنة وسيط واحد على تمثيلات النموذج (مشكلة مشابهة لـ Data Mixing في الفصل 18 لكن أكثر حدّة عبر الوسائط).

### 🔴 ج) طبقة التعمّق: فهم الفيديو (Video Understanding) — الوسيط الرابع

الفيديو = تسلسل صور + محور زمني إضافي، وهذا يفرض تحديًا حسابيًا كبيرًا (عدد الـ tokens يتفجّر بسرعة).

**استراتيجيات معالجة الفيديو الشائعة**:

| الاستراتيجية | الوصف | المقايضة |
|---|---|---|
| **أخذ عينات إطارات متباعدة (Frame Sampling)** | اختيار N إطار فقط (مثلاً 8-32) من الفيديو | فقدان تفاصيل زمنية دقيقة |
| **تجميع مكاني-زمني (Spatiotemporal Pooling)** | دمج عدة إطارات متتالية في تمثيل واحد مضغوط | يحافظ على الحركة بكفاءة أعلى |
| **Token Merging عبر الزمن** | دمج patches متشابهة عبر إطارات متتالية (تقليل التكرار) | يحافظ على كل من الدقة والكفاءة |
| **ترميز هرمي (Hierarchical)** | تمثيل قصير المدى محلي + تمثيل طويل المدى عام | الأقرب لما تستخدمه Gemini 1.5 لفهم فيديوهات الساعات |


In [ ]:
def sample_video_frames(video_tensor, num_frames=16, strategy="uniform"):
    """
    video_tensor: [total_frames, C, H, W]
    """
    total_frames = video_tensor.shape[0]
    if strategy == "uniform":
        indices = torch.linspace(0, total_frames - 1, num_frames).long()
    elif strategy == "keyframe":
        # اختيار الإطارات ذات أكبر تغيّر بصري (مبسّط)
        diffs = torch.abs(video_tensor[1:] - video_tensor[:-1]).mean(dim=(1, 2, 3))
        indices = torch.topk(diffs, num_frames).indices.sort().values
    return video_tensor[indices]


## 4. مثال محلول خطوة بخطوة

لنصمّم مفردات موحّدة صغيرة:

| النطاق | الوسيط | الحجم |
|---|---|---|
| 0 – 31,999 | نص | 32,000 |
| 32,000 – 40,959 | صورة (VQ codebook) | 8,960 |
| 40,960 – 48,959 | صوت (RVQ level 1) | 8,000 |
| 48,960 – 49,023 | رموز خاصة (`<image>`, `<audio>`, `<eos>`) | 64 |

**تسلسل مثال:**


```text
[<bos>] نص: "صف الصورة" [<image>] 32,145 38,902 … [<audio>] 41,220 … [<eos>]
```


**فكّ التسلسل:** نقرأ كل رمز ونوجّهه إلى decoderه: النص إلى tokenizer النص، الصورة إلى VQ decoder، الصوت إلى EnCodec decoder. هذا هو جوهر "Any-to-Any": Transformer واحد يتعلّم الانتقالات بين النطاقات، بينما decoders منفصلة تُعيد كل نطاق إلى شكله الأصلي.

**التحدي:** إذا كانت 90% من الرموز نصًا و10% صورة، سيهيمن النص. الحل: data mixing متوازن، modality dropout، وأوزان خسارة حسب النطاق.


## 5. نصائح وحيل

> 💡 **ابدأ بـ Late Fusion للفهم**: encoder صور قوي + adapter + LLM مُدرَّب مسبقًا أسرع وأرخص من تدريب نموذج موحّد من الصفر.

> 💡 **وازن البيانات صراحةً**: حدّد نسبة النص/الصورة/الصوت في كل دفعة، وراقب loss كل نطاق على حدة.

> 💡 **استخدم رموزًا خاصة واضحة**: `<image>`, `</image>`, `<audio>` تمنع الالتباس وتسهّل التقييم.

> 💡 **قلّل دقة الصور/الفيديو قبل التوكنة**: 224×224 أو 336×336 غالبًا كافية؛ الدقة العالية تُفجّر عدد الرموز.

> 💡 **قيّم كل نطاق على حدة**: لا يكفي متوسط واحد؛ قِس فهم الصورة، ASR، TTS، والاتساق عبر الوسائط.


## 6. أخطاء شائعة ومصائد

> ⚠️ **لصق رموز الوسائط دون حدود**: بدون رموز بداية/نهاية ونطاقات vocab واضحة، يفشل النموذج في معرفة أين تنتهي الصورة.

> ⚠️ **إهمال موازنة البيانات**: نموذج يتدرّب على نصوص أكثر بكثير من الصور قد يتجاهل الصور تمامًا. راقب النسب وطبّق sampling.

> ⚠️ **توقّع أن Early Fusion أفضل دائمًا**: Chameleon-style قوي لكنه مكلف ويحتاج بيانات ضخمة؛ Late Fusion قد يتفوق في الفهم بميزانية أقل.

> ⚠️ **تجاهل تفجّر رموز الفيديو**: فيديو 10 ثوانٍ بـ16fps قد يستهلك آلاف الرموز. استخدم frame sampling وtemporal pooling.

> ⚠️ **الحكم من عرض توضيحي واحد**: التقييم الموحّد صعب؛ استخدم MMMU/MMBench واختبارات عربية خاصة.


## 7. جدول مقارنة

| النموذج | الجهة | الوسائط المدعومة | الأسلوب |
|---|---|---|---|
| **Gemini 1.5** | Google DeepMind | نص، صورة، صوت، فيديو، كود | موحّد native منذ التصميم، نافذة سياق ضخمة (1M+ token) |
| **GPT-4o** | OpenAI | نص، صورة، صوت (end-to-end) | نموذج واحد native عبر الوسائط، زمن استجابة منخفض جدًا |
| **Chameleon** | Meta | نص، صورة (توليد وفهم) | رموز منفصلة موحّدة بالكامل (early fusion حقيقي) |
| **CoDi / CoDi-2** | Microsoft | نص، صورة، صوت، فيديو | توليد متزامن متعدد الوسائط عبر diffusion مشترك |
| **Unified-IO 2** | AI2 | نص، صورة، صوت، فيديو، إجراءات (actions) | تدريب من الصفر على كل الوسائط معًا |


## 8. ملخص أفضل الممارسات

- ✅ ابدأ دائمًا بـ Vision/Audio Encoder مُدرَّب مسبقًا ومجمّد، وأضف المحاذاة تدريجيًا (نهج LLaVA)، لا تُدرّب كل شيء من الصفر إلا إذا كان لديك موارد هائلة.
- ✅ افصل مرحلة Alignment (تعلّم الربط) عن مرحلة Instruction Tuning دائمًا.
- ✅ استخدم Classifier-Free Guidance في أي نظام توليد مشروط لضمان اتّباع الشرط النصي.
- ✅ لا تتجاهل تقييم الهلوسة البصرية (POPE) — فهي مشكلة شائعة جدًا في VLM.
- ✅ عند التعامل مع الفيديو، خطّط لاستراتيجية ضغط الرموز الزمنية منذ البداية؛ التطبيق الساذج (كل إطار = patches كاملة) يُفجّر طول السياق بسرعة.
- ✅ راقب نسب خلط البيانات بين الوسائط كما تراقبها بين مصادر النص (الفصل 18).

---


## 9. أسئلة مقابلات

**س1: ما الفرق بين Early Fusion وLate Fusion؟**

**الإجابة النموذجية:** Early Fusion يُدخل كل الوسائط كرموز في نفس التسلسل منذ البداية، فيتعلّم Transformer تفاعلات حرة بينها (Chameleon). Late/Modular Fusion يستخدم encoders منفصلة لكل وسيط ثم adapter يُدخل تمثيلاتها إلى LLM (LLaVA). الأول أقوى نظريًا في التكامل لكنه مكلف ويحتاج بيانات ضخمة؛ الثاني أسرع وأرخص ويستفيد من نماذج جاهزة.

**س2: كيف تمنع هيمنة النص على الصوت/الصورة؟**

**الإجابة النموذجية:** عبر data mixing متوازن، sampling حسب النطاق، modality dropout، وأوزان خسارة منفصلة. راقب loss وaccuracy كل نطاق؛ إذا تدهور نطاق، ارفع نسبته مؤقتًا أو جمّد بعض الطبقات.

**س3: لماذا يعد تقييم Any-to-Any أصعب من تقييم LLM نصي؟**

**الإجابة النموذجية:** لأن الإخراج متعدد الوسائط ولا توجد إجابة مرجعية واحدة؛ تحتاج مقاييس لكل نطاق (WER للصوت، FID/CLIPScore للصور، دقة نصية) plus تقييمًا بشريًا للاتساق. كما أن التلوث عبر الوسائط والتحيزات يختلفان.


## 10. تمرين عملي

1. صمّم `vocab_ranges` كاملة (نص/صورة/صوت/فيديو/خاص) واكتب دالة `decode_unified_sequence`.
2. خذ تسلسلًا يحتوي نصًا وصورة، وفكّكه إلى مكوّناته.
3. احسب عدد رموز فيديو 10 ثوانٍ بـ16fps ودقة 224×224 مع patch 16×16، ثم اقترح طريقتين لتقليله.
4. صمّم خطة data mixing لثلاثة وسائط بنسب 70/20/10، واشرح كيف تراقب التوازن.
5. اكتب تقييمًا نقديًا لنموذج Any-to-Any افتراضي: 3 نقاط قوة و3 مخاطر.


## 11. مراجع للتعمّق

- Chameleon Team (2024). *Chameleon: Mixed-Modal Early-Fusion Foundation Models*. Meta.
- Gemini Team (2024). *Gemini 1.5: Unlocking Multimodal Understanding Across Millions of Tokens of Context*.
- OpenAI (2024). *GPT-4o System Card*.
- Lu, J., et al. (2022). *Unified-IO: A Unified Model for Vision, Language, and Multi-Modal Tasks*.
- Sun, Q., et al. (2023). *Generative Pretraining in Multimodality* (Emu).


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النماذج متعددة الوسائط / الفصل 40: النماذج الموحّدة: Any-to-Any.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
